# Imports

In [ ]:
from dotenv import load_dotenv
from anthropic import Anthropic
from anthropic.types import ToolParam
import json
load_dotenv()

# Create the Anthropic Client

In [ ]:
client = Anthropic()
model = "claude-haiku-4-5"

# Text Editor Tool Implementation

In [ ]:
# Implementation of the TextEditorTool
import os
import shutil
from typing import Optional, List


class TextEditorTool:
    def __init__(self, base_dir: str = "", backup_dir: str = ""):
        self.base_dir = base_dir or os.getcwd()
        self.backup_dir = backup_dir or os.path.join(self.base_dir, ".backups")
        os.makedirs(self.backup_dir, exist_ok=True)

    def _validate_path(self, file_path: str) -> str:
        abs_path = os.path.normpath(os.path.join(self.base_dir, file_path))
        if not abs_path.startswith(self.base_dir):
            raise ValueError(
                f"Access denied: Path '{file_path}' is outside the allowed directory"
            )
        return abs_path

    def _backup_file(self, file_path: str) -> str:
        if not os.path.exists(file_path):
            return ""
        file_name = os.path.basename(file_path)
        backup_path = os.path.join(
            self.backup_dir, f"{file_name}.{os.path.getmtime(file_path):.0f}"
        )
        shutil.copy2(file_path, backup_path)
        return backup_path

    def _restore_backup(self, file_path: str) -> str:
        file_name = os.path.basename(file_path)
        backups = [
            f for f in os.listdir(self.backup_dir) if f.startswith(file_name + ".")
        ]
        if not backups:
            raise FileNotFoundError(f"No backups found for {file_path}")

        latest_backup = sorted(backups, reverse=True)[0]
        backup_path = os.path.join(self.backup_dir, latest_backup)

        shutil.copy2(backup_path, file_path)
        return f"Successfully restored {file_path} from backup"

    def _count_matches(self, content: str, old_str: str) -> int:
        return content.count(old_str)

    def view(self, file_path: str, view_range: Optional[List[int]] = None) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if os.path.isdir(abs_path):
                try:
                    return "\n".join(os.listdir(abs_path))
                except PermissionError:
                    raise PermissionError(
                        "Permission denied. Cannot list directory contents."
                    )

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            if view_range:
                start, end = view_range
                lines = content.split("\n")

                if end == -1:
                    end = len(lines)

                selected_lines = lines[start - 1 : end]

                result = []
                for i, line in enumerate(selected_lines, start):
                    result.append(f"{i}: {line}")

                return "\n".join(result)
            else:
                lines = content.split("\n")
                result = []
                for i, line in enumerate(lines, 1):
                    result.append(f"{i}: {line}")

                return "\n".join(result)

        except UnicodeDecodeError:
            raise UnicodeDecodeError(
                "utf-8",
                b"",
                0,
                1,
                "File contains non-text content and cannot be displayed.",
            )
        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot access file.")
        except Exception as e:
            raise type(e)(str(e))

    def str_replace(self, file_path: str, old_str: str, new_str: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            match_count = self._count_matches(content, old_str)

            if match_count == 0:
                raise ValueError(
                    "No match found for replacement. Please check your text and try again."
                )
            elif match_count > 1:
                raise ValueError(
                    f"Found {match_count} matches for replacement text. Please provide more context to make a unique match."
                )

            # Create backup before modifying
            self._backup_file(abs_path)

            # Perform the replacement
            new_content = content.replace(old_str, new_str)

            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(new_content)

            return "Successfully replaced text at exactly one location."

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def create(self, file_path: str, file_text: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            # Check if file already exists
            if os.path.exists(abs_path):
                raise FileExistsError(
                    "File already exists. Use str_replace to modify it."
                )

            # Create parent directories if they don't exist
            os.makedirs(os.path.dirname(abs_path), exist_ok=True)

            # Create the file
            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(file_text)

            return f"Successfully created {file_path}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot create file.")
        except Exception as e:
            raise type(e)(str(e))

    def insert(self, file_path: str, insert_line: int, new_str: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            # Create backup before modifying
            self._backup_file(abs_path)

            with open(abs_path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            # Handle line endings
            if lines and not lines[-1].endswith("\n"):
                new_str = "\n" + new_str

            # Insert at the beginning if insert_line is 0
            if insert_line == 0:
                lines.insert(0, new_str + "\n")
            # Insert after the specified line
            elif insert_line > 0 and insert_line <= len(lines):
                lines.insert(insert_line, new_str + "\n")
            else:
                raise IndexError(
                    f"Line number {insert_line} is out of range. File has {len(lines)} lines."
                )

            with open(abs_path, "w", encoding="utf-8") as f:
                f.writelines(lines)

            return f"Successfully inserted text after line {insert_line}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))


# Example 1

In [ ]:
messages = [
    {
        "role": "user",
        "content": "Open the ./main.py file and summarize its contents. Make sure to include ./ for current directory paths"
    }
]

In [ ]:
response = \
    client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = messages,
    tools=[{
            "type": "text_editor_20250728",
            "name": "str_replace_based_edit_tool",
        }]
    )


In [ ]:
for element in response.content:
    print(element, "\n")

In [ ]:
text_editor_tool = TextEditorTool()
tool_result = text_editor_tool.view("./main.py")
print(tool_result)

In [ ]:
messages = [
    {
        "role": "user",
        "content": "Open the ./main.py file and summarize its contents. Make sure to include ./ for current directory paths"
    },
    {
        "role": "assistant",
        "content": response.content
    },
    {
        "role": "user",
        "content":
        [
            {
                "type": "tool_result",
                "tool_use_id": response.content[1].id,
                "content": json.dumps(tool_result),
                "is_error": False
            }
        ]
    },
]

In [ ]:
for message in messages:
    print(message, "\n")

In [ ]:
response = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = messages,
    tools=[{
            "type": "text_editor_20250728",
            "name": "str_replace_based_edit_tool",
        }]
    )

In [ ]:
for element in response.content:
    print(element, "\n")

# Example 2

In [ ]:
messages = [
    {
        "role": "user",
        "content": """
        Open the ./main.py file and write out a function to calculate pi to the 5th digit.

        Then create a ./test.py file to test your implementation.

        Make sure to include ./ for current directory paths
        """
    }
]

In [ ]:
response_1 = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = messages,
    tools=[{
            "type": "text_editor_20250728",
            "name": "str_replace_based_edit_tool",
        }]
    )

In [15]:
for element in response_1.content:
    print(element, "\n")

TextBlock(citations=None, text="I'll help you create a function to calculate pi to the 5th digit and then test it. Let me start by viewing the current directory and then create the necessary files.", type='text') 

ToolUseBlock(id='toolu_01WUNfjgrSVrWQnZr239sxGr', caller=DirectCaller(type='direct'), input={'command': 'view', 'path': './'}, name='str_replace_based_edit_tool', type='tool_use') 



In [16]:
text_editor_tool = TextEditorTool()
tool_result_1 = text_editor_tool.view(response_1.content[1].input["path"])
print(tool_result_1)

.backups
09_text_edit_tool.ipynb
main.py
test.py


In [17]:
messages = [
    {
        "role": "user",
        "content": """
        Open the ./main.py file and write out a function to calculate pi to the 5th digit.

        The create a ./test.py file to test your implementation.

        Make sure to include ./ for current directory paths
        """
    },
    {
        "role": "assistant",
        "content": response_1.content
    },
    {
        "role": "user",
        "content":
        [
            {
                "type": "tool_result",
                "tool_use_id": response_1.content[1].id,
                "content": json.dumps(tool_result_1),
                "is_error": False
            }
        ]
    },
]

In [18]:
response_2 = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = messages,
    tools=[{
            "type": "text_editor_20250728",
            "name": "str_replace_based_edit_tool",
        }]
    )

In [19]:
for element in response_2.content:
    print(element, "\n")

TextBlock(citations=None, text='Let me first check the current content of main.py:', type='text') 

ToolUseBlock(id='toolu_01KH6geF67rzAuRt9TxcK99C', caller=DirectCaller(type='direct'), input={'command': 'view', 'path': './main.py'}, name='str_replace_based_edit_tool', type='tool_use') 



In [20]:
text_editor_tool = TextEditorTool()
tool_result_2 = text_editor_tool.view(response_2.content[1].input["path"])
print(tool_result_2)

1: def greeting():
2:     print("Hi there!")
3: 


In [21]:
messages = [
    {
        "role": "user",
        "content": """
        Open the ./main.py file and write out a function to calculate pi to the 5th digit.

        The create a ./test.py file to test your implementation.

        Make sure to include ./ for current directory paths
        """
    },
    {
        "role": "assistant",
        "content": response_1.content
    },
    {
        "role": "user",
        "content":
        [
            {
                "type": "tool_result",
                "tool_use_id": response_1.content[1].id,
                "content": json.dumps(tool_result_1),
                "is_error": False
            }
        ]
    },
    {
        "role": "assistant",
        "content": response_2.content
    },
    {
        "role": "user",
        "content":
        [
            {
                "type": "tool_result",
                "tool_use_id": response_2.content[1].id,
                "content": json.dumps(tool_result_2),
                "is_error": False
            }
        ]
    },
]

In [22]:
response_3 = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = messages,
    tools=[{
            "type": "text_editor_20250728",
            "name": "str_replace_based_edit_tool",
        }]
    )

In [23]:
for element in response_3.content:
    print(element, "\n")

TextBlock(citations=None, text="Now I'll update the main.py file to include a function that calculates pi to the 5th digit:", type='text') 

ToolUseBlock(id='toolu_01UzVeQmxjVF5KuzYu7gL6R5', caller=DirectCaller(type='direct'), input={'command': 'str_replace', 'path': './main.py', 'old_str': 'def greeting():\n    print("Hi there!")', 'new_str': 'def greeting():\n    print("Hi there!")\n\n\ndef calculate_pi():\n    """\n    Calculate pi to the 5th digit using the Machin formula.\n    Returns pi approximately as 3.14159\n    """\n    from decimal import Decimal, getcontext\n    \n    # Set precision to 10 decimal places to ensure accuracy to the 5th digit\n    getcontext().prec = 10\n    \n    # Using the Machin formula: pi/4 = 4*arctan(1/5) - arctan(1/239)\n    # This converges quickly and gives us good precision\n    \n    one = Decimal(1)\n    five = Decimal(5)\n    two_three_nine = Decimal(239)\n    \n    # Calculate arctan using Taylor series\n    def arctan(x, num_terms=50):\n      

In [24]:
text_editor_tool = TextEditorTool()
tool_result_3 = text_editor_tool.str_replace(
    response_3.content[1].input["path"],
    response_3.content[1].input["old_str"],
    response_3.content[1].input["new_str"]
)
print(tool_result_3)

Successfully replaced text at exactly one location.


In [25]:
messages = [
    {
        "role": "user",
        "content": """
        Open the ./main.py file and write out a function to calculate pi to the 5th digit.

        The create a ./test.py file to test your implementation.

        Make sure to include ./ for current directory paths
        """
    },
    {
        "role": "assistant",
        "content": response_1.content
    },
    {
        "role": "user",
        "content":
        [
            {
                "type": "tool_result",
                "tool_use_id": response_1.content[1].id,
                "content": json.dumps(tool_result_1),
                "is_error": False
            }
        ]
    },
    {
        "role": "assistant",
        "content": response_2.content
    },
    {
        "role": "user",
        "content":
        [
            {
                "type": "tool_result",
                "tool_use_id": response_2.content[1].id,
                "content": json.dumps(tool_result_2),
                "is_error": False
            }
        ]
    },{
        "role": "assistant",
        "content": response_3.content
    },
    {
        "role": "user",
        "content":
        [
            {
                "type": "tool_result",
                "tool_use_id": response_3.content[1].id,
                "content": json.dumps(tool_result_3),
                "is_error": False
            }
        ]
    },
]

In [26]:
response_4 = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = messages,
    tools=[{
            "type": "text_editor_20250728",
            "name": "str_replace_based_edit_tool",
        }]
    )

In [27]:
for element in response_4.content:
    print(element, "\n")

TextBlock(citations=None, text='Now let me create the test.py file to test the implementation:', type='text') 

ToolUseBlock(id='toolu_01J2UhThh1gaSwYzsdTVFaAM', caller=DirectCaller(type='direct'), input={'command': 'create', 'path': './test.py', 'file_text': 'import sys\nfrom main import calculate_pi\n\n\ndef test_calculate_pi():\n    """\n    Test the calculate_pi function to verify it returns pi to the 5th digit.\n    Pi to the 5th digit is: 3.14159\n    """\n    result = calculate_pi()\n    \n    print(f"Calculated pi: {result}")\n    print(f"Calculated pi (to 5 decimals): {result:.5f}")\n    \n    # Expected value of pi to the 5th digit\n    expected = 3.14159\n    \n    # Check if the result matches to the 5th digit\n    result_rounded = round(result, 5)\n    \n    print(f"\\nExpected pi (to 5 decimals): {expected}")\n    print(f"Result rounded to 5 decimals: {result_rounded}")\n    \n    # Verify the result\n    if result_rounded == expected:\n        print("\\n✓ Test PASSED! Pi

In [29]:
text_editor_tool = TextEditorTool()
tool_result_4 = text_editor_tool.create(
    response_4.content[1].input["path"],
    response_4.content[1].input["file_text"]
)
print(tool_result_4)

Successfully created ./test.py
